[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/corrections/seance3_correction.ipynb)

# Séance 4.3 — Arbres de décision — comprendre les variables qui déterminent la prédiction

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- lire un arbre de décision comme une suite de règles métier
- choisir la profondeur d'un arbre par validation croisée, sans toucher au test
- dire pourquoi l'importance native d'un arbre est biaisée
- mesurer l'importance d'une variable par permutation, sur le jeu de test
- appliquer cette mesure à n'importe quel modèle, arbre ou régression
- montrer le **sens** d'un effet avec une dépendance partielle

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, f1_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")   ## texte -> nombre
tel = tel.dropna(subset=["total"])   ## 11 abonnes sans facture cumulee

y = tel["churn"]
# .astype(float) : les dependances partielles refusent les colonnes
# entieres, et les 0/1 de get_dummies sont des booleens
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True).astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(X.shape[1], "variables |", len(X_tr), "abonnes d'apprentissage")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — L'arbre et ses règles

> **Votre mission :**
> - Ajuster un `DecisionTreeClassifier` de profondeur 3 (`random_state=42`) → `a`.
> - Afficher ses règles avec `export_text`.
> - Mettre le nom de la variable de la **première coupure** dans `premiere`.

In [ ]:
a = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)

print(export_text(a, feature_names=list(X.columns)))   ## les regles, en clair

# La toute premiere ligne de l'arbre porte sur le contrat mensuel
premiere = "contrat_mensuel"

In [ ]:
verifier("1 - premiere coupure", premiere == "contrat_mensuel",
         "regardez la premiere ligne de export_text")

### Exercice 2 — Noter deux arbres

> **Votre mission :**
> - Ajuster un arbre de profondeur 4 → `bon`, et un arbre **sans aucune limite** → `libre` (`random_state=42` pour les deux).
> - Calculer leur F1 **de test** → `f1_bon` et `f1_libre`, arrondis à 3 décimales.
> - Relever aussi le F1 de `libre` **en apprentissage** → `f1_libre_tr`. Que constatez-vous ?

In [ ]:
bon = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
libre = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)   ## aucune limite

f1_bon = round(f1_score(y_te, bon.predict(X_te)), 3)
f1_libre = round(f1_score(y_te, libre.predict(X_te)), 3)
f1_libre_tr = round(f1_score(y_tr, libre.predict(X_tr)), 3)
print("bride", f1_bon, "| libre", f1_libre, "| libre en apprentissage", f1_libre_tr)

# 0,993 en apprentissage contre 0,501 en test : l'arbre libre a memorise
# les 4 922 abonnes qu'on lui a montres. Exactement la seance 4.1.

In [ ]:
verifier("2a - F1 de l'arbre bride", abs(f1_bon - 0.523) < 0.02, "max_depth=4")
verifier("2b - F1 de l'arbre libre", abs(f1_libre - 0.501) < 0.02,
         "predisez sur X_te, pas sur X_tr")
verifier("2c - F1 de l'arbre libre en apprentissage", f1_libre_tr > 0.9,
         "un arbre sans limite reproduit presque parfaitement ce qu'il a vu")

### Exercice 3 — La bonne profondeur, sans toucher au test

> **Votre mission :**
> - Par validation croisée à 5 plis sur l'**apprentissage**, comparer les profondeurs 2, 4 et 8.
> - Mettre le F1 moyen de la profondeur 4 dans `f1_cv4` (3 décimales).

In [ ]:
# La validation croisee travaille sur l'APPRENTISSAGE : le test reste ferme
for prof in [2, 4, 8]:   ## trois reglages a departager
    s = cross_val_score(DecisionTreeClassifier(max_depth=prof, random_state=42),
                        X_tr, y_tr, cv=5, scoring="f1")
    print("profondeur", prof, ":", round(s.mean(), 3))

f1_cv4 = round(cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    X_tr, y_tr, cv=5, scoring="f1").mean(), 3)

# 0,586 / 0,558 / 0,546 : trois centiemes d'ecart. La question 13 montre
# que c'est aussi l'ordre de grandeur du bruit entre les cinq plis.

In [ ]:
verifier("3 - F1 croise a la profondeur 4", abs(f1_cv4 - 0.558) < 0.02,
         "cross_val_score sur X_tr et y_tr, jamais sur le test")

### Exercice 4 — L'importance native

> **Votre mission :**
> - Extraire l'importance native de l'arbre **sans limite** dans une `Series` indexée par le nom des colonnes → `native`.
> - Mettre la variable en tête du classement dans `top_native`.

In [ ]:
native = pd.Series(libre.feature_importances_, index=X.columns)   ## le natif

top_native = native.idxmax()   ## "mensuel", le montant de la facture
print(native.sort_values(ascending=False).head(4).round(3))

# Les deux factures occupent les deux premieres places. Retenez-le :
# l'exercice 5 le demonte.

In [ ]:
verifier("4 - variable la plus importante (native)", top_native == "mensuel",
         "l'attribut s'appelle feature_importances_, avec un underscore final")

### Exercice 5 — L'importance par permutation

> **Votre mission :**
> - Mesurer l'importance par permutation du **même arbre**, sur le **jeu de test** (`n_repeats=5`, `random_state=42`, `scoring='f1'`) → `perm`.
> - Mettre la variable arrivée en tête dans `top_perm`.
> - Comparez à l'exercice 4.

In [ ]:
# Sur le TEST : c'est ce que le modele perd sur des donnees inconnues
pi = permutation_importance(libre, X_te, y_te, n_repeats=5,
                            random_state=42, scoring="f1")
perm = pd.Series(pi.importances_mean, index=X.columns)

top_perm = perm.idxmax()   ## "contrat_mensuel" : le classement est renverse
print(perm.sort_values(ascending=False).head(4).round(4))

# contrat_mensuel arrive premier, loin devant. Les deux factures reculent.

In [ ]:
verifier("5 - variable la plus importante (permutation)", top_perm == "contrat_mensuel",
         "permutation_importance(libre, X_te, y_te, ...)")

### Exercice 6 — Le rang qui bascule

> **Votre mission :**
> - Quel est le **rang** de `contrat_mensuel` dans chacun des deux classements ?
> - Mettre les deux rangs dans `rang_native` et `rang_perm` (1 = première).
> - C'est la variable sur laquelle on peut agir : se tromper de classement coûte cher.

In [ ]:
ordre_native = native.sort_values(ascending=False).index
ordre_perm = perm.sort_values(ascending=False).index

rang_native = list(ordre_native).index("contrat_mensuel") + 1   ## +1 : rang 1, pas 0
rang_perm = list(ordre_perm).index("contrat_mensuel") + 1
print("contrat_mensuel : rang", rang_native, "en natif,", rang_perm, "en permutation")

# Troisieme en natif, PREMIERE en permutation. Une note de direction fondee
# sur l'importance native aurait vise la facture — sur laquelle on ne peut
# rien — au lieu du contrat, qu'on peut changer des demain.

In [ ]:
verifier("6a - rang natif du contrat mensuel", rang_native == 3,
         "cherchez sa position dans ordre_native")
verifier("6b - rang par permutation", rang_perm == 1,
         "elle arrive en tete du classement par permutation")

### Exercice 7 — La même mesure sur un tout autre modèle

> **Votre mission :**
> - `feature_importances_` n'existe que pour les arbres. La permutation, elle, marche partout.
> - Remonter la régression logistique de la séance 4.2 → `logi`, puis mesurer l'importance par permutation sur le test → `perm_logi`.
> - Mettre la variable en tête dans `top_logi`. Retrouve-t-on la même réponse qu'à l'exercice 5 ?

In [ ]:
logi = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logi.fit(X_tr, y_tr)

# Exactement la meme fonction, sur un modele qui n'est pas un arbre
pl = permutation_importance(logi, X_te, y_te, n_repeats=5,
                            random_state=42, scoring="f1")
perm_logi = pd.Series(pl.importances_mean, index=X.columns)

top_logi = perm_logi.idxmax()   ## "anc"
print(perm_logi.sort_values(ascending=False).head(4).round(4))

# anc, contrat_mensuel et internet_fibre en tete, comme sur l'arbre bride.
# Deux familles de modeles sans rapport, un meme trio : c'est ce qui rend
# une recommandation solide.

In [ ]:
verifier("7a - variable en tete pour la logistique", top_logi == "anc",
         "idxmax() sur perm_logi")
verifier("7b - le contrat suit de pres", perm_logi.nlargest(3).index[1] == "contrat_mensuel",
         "regardez les trois premieres du classement")

### Exercice 8 — Le sens de l'effet

> **Votre mission :**
> - Tracer la dépendance partielle de l'arbre bridé `bon` pour l'ancienneté (`anc`).
> - Mettre dans `sens` le mot « baisse » ou « monte », selon ce que fait le risque quand l'ancienneté augmente.
> - Puis vérifier sur les taux réels par tranche d'ancienneté → `taux_1ere_annee` (en %, 1 décimale).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(bon, X_te, ["anc"], ax=ax)   ## anc seule
plt.show()

# La courbe descend par paliers sur les premiers mois
sens = "baisse"

tranches = pd.cut(tel["anc"], [0, 12, 24, 48, 72])   ## comme au bloc 2
taux = (tel.groupby(tranches, observed=True)["churn"].mean() * 100).round(1)
taux_1ere_annee = taux.iloc[0]   ## 47,7 %
print(taux)

# 47,7 % la premiere annee, 9,5 % au-dela de quatre ans : le modele et les
# donnees brutes racontent la meme histoire.

In [ ]:
verifier("8a - sens de l'effet de l'anciennete", sens == "baisse",
         "regardez la courbe : le risque augmente-t-il avec l'anciennete ?")
verifier("8b - churn de la premiere annee", abs(taux_1ere_annee - 47.7) < 0.5,
         "la premiere tranche du decoupage, 0 a 12 mois")

### Exercice 9 — Question de synthèse

> **Votre mission :**
> - Le comité veut **une** action, chiffrée.
> - Calculer le taux de départ des contrats mensuels et celui des engagements deux ans → `t_mensuel` et `t_deux_ans` (en %, 1 décimale).
> - En déduire l'écart en points → `ecart_points`. Puis rédigez la recommandation en commentaire.

In [ ]:
taux_contrat = (tel.groupby("contrat")["churn"].mean() * 100).round(1)

t_mensuel = taux_contrat["mensuel"]     ## 42,7 %
t_deux_ans = taux_contrat["deux_ans"]   ## 2,8 %
ecart_points = round(t_mensuel - t_deux_ans, 1)   ## quarante points
print(t_mensuel, "% contre", t_deux_ans, "% ->", ecart_points, "points")

# Recommandation possible :
# "Le premier facteur de depart n'est ni la facture ni l'age du client :
#  c'est le type de contrat. Un abonne au mois resilie a 42,7 %, un abonne
#  engage deux ans a 2,8 % — quarante points d'ecart. L'action a tester
#  est une incitation a l'engagement sur les douze premiers mois, periode
#  ou se concentre pres de la moitie des departs."

In [ ]:
verifier("9a - churn des contrats mensuels", t_mensuel == 42.7, "groupby('contrat')")
verifier("9b - ecart en points", abs(ecart_points - 39.9) < 0.2,
         "la modalite s'ecrit deux_ans")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 10 — Fabriquer le biais soi-même

> **Votre mission :**
> - Ajouter à `X` une colonne `bruit` remplie de nombres **au hasard**, sans aucun lien avec la cible.
> - Réajuster l'arbre **sans limite** et regarder où cette colonne se classe dans chacune des deux importances.
> - *Nouveau :* `np.random.default_rng(42).random(len(X))` fabrique des nombres au hasard.

In [ ]:
bruite = X.copy()
bruite["bruit"] = np.random.default_rng(42).random(len(X))   ## zero information

b_tr, b_te, c_tr, c_te = train_test_split(
    bruite, y, test_size=0.3, random_state=42, stratify=y)
lb = DecisionTreeClassifier(random_state=42).fit(b_tr, c_tr)

nat = pd.Series(lb.feature_importances_, index=bruite.columns)
pib = permutation_importance(lb, b_te, c_te, n_repeats=5,
                             random_state=42, scoring="f1")
per = pd.Series(pib.importances_mean, index=bruite.columns)

print("rang du bruit en natif      :", int(nat.rank(ascending=False)["bruit"]))
print("rang du bruit en permutation:", int(per.rank(ascending=False)["bruit"]))

# PREMIERE en natif. Une colonne qui ne contient rien, absolument rien,
# arrive en tete du classement d'importance — parce qu'elle prend 7 032
# valeurs distinctes, donc offre 7 031 coupures possibles, et que l'arbre
# libre s'en sert a tous les etages pour isoler ses derniers individus.
# La permutation la renvoie au huitieme rang. C'est la demonstration la
# plus nette du biais, et la raison de ne jamais citer l'importance native
# telle quelle.

### Question 11 — Une importance nulle ne veut pas dire « inutile »

> **Votre mission :**
> - Ajouter une colonne `anc_bis`, copie exacte de `anc`. Réajuster l'arbre de profondeur 4.
> - Regarder l'importance native **et** l'importance par permutation des deux colonnes.
> - `anc_bis` est à zéro partout. Est-elle sans information pour autant ? Quelle précaution en tirer ?

In [ ]:
double = X.copy()
double["anc_bis"] = double["anc"]   ## copie exacte : information dupliquee

d_tr, d_te, e_tr, e_te = train_test_split(
    double, y, test_size=0.3, random_state=42, stratify=y)
fd = DecisionTreeClassifier(max_depth=4, random_state=42).fit(d_tr, e_tr)

nat = pd.Series(fd.feature_importances_, index=double.columns)
pid = permutation_importance(fd, d_te, e_te, n_repeats=5,
                             random_state=42, scoring="f1")
per = pd.Series(pid.importances_mean, index=double.columns)
print(pd.DataFrame({"native": nat, "permutation": per}).loc[["anc", "anc_bis"]].round(4))

# anc_bis est a ZERO dans les deux classements, alors qu'elle contient
# exactement la meme information que anc. L'arbre a simplement choisi
# l'une des deux et n'a jamais regarde l'autre ; melanger une colonne dont
# il ne se sert pas ne lui coute rien.
# Precaution : une importance de zero dit "ce modele-la ne s'en sert pas",
# jamais "cette variable ne sert a rien". Avant de supprimer une colonne
# sur la foi d'un classement, regarder les correlations.

### Question 12 — L'importance dépend de la mesure choisie

> **Votre mission :**
> - Refaire l'importance par permutation de l'arbre bridé avec `scoring='accuracy'` au lieu de `scoring='f1'`.
> - Les valeurs changent-elles ? Et le classement ?
> - Que faut-il donc préciser en annonçant une importance ?

In [ ]:
for mesure in ["f1", "accuracy"]:
    pi = permutation_importance(bon, X_te, y_te, n_repeats=5,
                                random_state=42, scoring=mesure)
    q = pd.Series(pi.importances_mean, index=X.columns)
    print(mesure, ":", q.nlargest(3).round(4).to_dict())

# Les VALEURS sont quatre fois plus petites avec la justesse : melanger
# une colonne fait perdre beaucoup de F1 et peu de justesse, puisque la
# justesse est deja tres elevee grace a la classe majoritaire (seance 4.2).
# Le CLASSEMENT bouge aussi : internet_fibre passe devant contrat_mensuel.
# Une importance par permutation n'est donc pas une propriete de la
# variable : c'est une perte, mesuree sur UNE metrique. On annonce toujours
# laquelle.

### Question 13 — La validation croisée est bruitée

> **Votre mission :**
> - Afficher les cinq scores individuels de la validation croisée aux profondeurs 4 et 5, et leur écart-type.
> - L'écart entre les deux moyennes est-il plus grand que ce bruit ?
> - Que faire quand une mesure ne départage pas deux réglages ?

In [ ]:
for prof in [4, 5]:
    s = cross_val_score(DecisionTreeClassifier(max_depth=prof, random_state=42),
                        X_tr, y_tr, cv=5, scoring="f1")
    print(f"profondeur {prof} : {s.round(3)}  moyenne {s.mean():.3f}  "
          f"ecart-type {s.std():.3f}")

# Les moyennes sont a 0,558 et 0,566, soit 0,008 d'ecart. La dispersion
# entre les cinq plis est de 0,022 et 0,043 : trois a cinq fois plus
# grande. Ces deux profondeurs ne sont donc PAS departageables — le
# classement observe tient au decoupage, pas au modele.
# Que faire : choisir sur un autre critere. La simplicite, la lisibilite,
# le cout de calcul. Meme raisonnement qu'a la seance 3.2 sur un ecart non
# significatif : ne pas conclure d'une difference qu'on ne sait pas voir.

### Question 14 — Un arbre par segment

> **Votre mission :**
> - Ajuster un arbre de profondeur 3 sur les seuls abonnés au contrat **mensuel**.
> - Ses règles sont-elles les mêmes que celles de l'arbre général ?
> - Qu'est-ce que ça dit de l'idée d'un modèle unique pour toute la clientèle ?

In [ ]:
mensuels = tel.query("contrat == 'mensuel'")   ## un seul segment
ym = mensuels["churn"]
Xm = pd.get_dummies(mensuels.drop(columns=["churn"]), drop_first=True)

m_tr, m_te, n_tr, n_te = train_test_split(
    Xm, ym, test_size=0.3, random_state=42, stratify=ym)
am = DecisionTreeClassifier(max_depth=3, random_state=42).fit(m_tr, n_tr)

print("taux de churn du segment :", round(100 * ym.mean(), 1), "%")
print(export_text(am, feature_names=list(Xm.columns)))

# Le contrat ayant disparu (tout le monde est au mensuel), l'arbre coupe
# sur autre chose : le type d'internet d'abord, l'anciennete ensuite.
# Segmenter d'abord, modeliser ensuite, revele des leviers que le modele
# global masquait — ici, la fibre.

### Question 15 — Expliquer UN client, avec SHAP

> **Votre mission :**
> - Les importances précédentes sont **globales** : elles décrivent le modèle, pas un client.
> - SHAP décompose une prédiction individuelle en contributions chiffrées.
> - Le paquet n'est pas fourni par Colab : la ligne d'installation vous est donnée, le reste est à écrire.
> - ⚠️ Environ **42 Mo** téléchargés par la machine virtuelle Colab — rien ne passe par votre connexion — et une trentaine de secondes de calcul.
> - Expliquer la prédiction du **premier abonné du jeu de test** : quelles variables la poussent à la hausse, lesquelles à la baisse ?

In [ ]:
!pip install -q shap
import shap

explainer = shap.TreeExplainer(bon)   ## l'arbre bride de la section 2
echantillon = X_te.iloc[:200]         ## 200 lignes suffisent, et c'est rapide
valeurs = explainer.shap_values(echantillon)
v = valeurs[:, :, 1] if np.array(valeurs).ndim == 3 else valeurs

client = pd.Series(v[0], index=X.columns)
print("probabilite de depart :", round(bon.predict_proba(echantillon.iloc[[0]])[0, 1], 3))
client.reindex(client.abs().sort_values(ascending=False).index).head(5).round(4)

# Positif = pousse au depart, negatif = retient. On peut enfin dire a un
# conseiller POURQUOI ce client-la est signale — et donc quoi lui
# proposer. C'est ce que ni l'importance ni la dependance partielle ne
# savent faire : elles decrivent le modele, pas la personne.

### Question 16 — Question de synthèse

> **Votre mission :**
> - Rédigez la note de direction qui répond à la question du comité : *« qu'est-ce qui fait partir nos clients ? »*
> - Contrainte : trois facteurs classés, chacun avec son **sens** et son ordre de grandeur, plus une phrase sur ce que ces données ne permettent **pas** d'affirmer.
> - Calculez d'abord les chiffres dont vous avez besoin.

In [ ]:
print("--- facteurs, par permutation sur le test (arbre bride)")
pi = permutation_importance(bon, X_te, y_te, n_repeats=5,
                            random_state=42, scoring="f1")
print(pd.Series(pi.importances_mean, index=X.columns).nlargest(3).round(4))
print("\n--- taux de depart par contrat")
print((tel.groupby("contrat")["churn"].mean() * 100).round(1))
print("\n--- taux de depart par tranche d'anciennete")
print((tel.groupby(pd.cut(tel["anc"], [0, 12, 24, 48, 72]), observed=True)["churn"]
       .mean() * 100).round(1))

# Note possible :
# "1. L'anciennete et le type de contrat arrivent en tete, au coude a
#     coude, et les deux mesures d'importance les designent sur deux
#     familles de modeles differentes.
#  2. Le contrat : 42,7 % de resiliations au mois contre 2,8 % a deux ans,
#     soit quarante points d'ecart.
#  3. L'anciennete : 47,7 % de departs la premiere annee, 9,5 % au-dela de
#     quatre ans. Le risque est divise par cinq, et il ne s'annule jamais.
#  Vient ensuite le type d'internet — la fibre — nettement devant les
#  montants de facture, qui ne pesent presque rien une fois le reste
#  pris en compte.
#  Limite : ce sont des ASSOCIATIONS mesurees sur une photographie du parc
#  a un instant donne, pas un suivi dans le temps. Rien n'etablit
#  qu'imposer un engagement de deux ans ferait baisser le churn : il se
#  peut que les clients deja fideles choisissent ce contrat. Seul un test
#  A/B trancherait — c'est l'objet du bloc 5."